# Capstone — Lane 3: Structured Content Archetype Clustering

Mirrors the deployed paper structure section-for-section. Self-contained: rebuilds from raw parquet like w05/w06/w07. Same mid-panel month, same confirmed schema, same exclusions throughout.

## 1. Question

Does an unsupervised content archetype (KMeans cluster) help predict whether a content item is an above- or below-median CTR performer, beyond what raw position/engagement signals already capture — and can a transparent, honestly-validated rule outperform a naive baseline on the same client-grouped split?

Unit of analysis: one content item, content-month grain (`month=2026-03`). Population: clients with full `gsc_and_ga4` access. This is decision-support for content-review prioritization, not a causal or revenue claim.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import json
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
FINAL_MONTH = "2026-06"
print("Connected. Question and scope stated above — no query needed for this section.")

## 2. Data

**Release**: FlyRank internship-warehouse (Hugging Face, gated). **Tables used**: `dim_content`, `dim_clients`, `fact_content_daily_performance` (partitioned by `month=YYYY-MM`). **Date window**: mid-panel month `2026-03`; final month `2026-06` used only to demonstrate leakage, never for feature/label construction. **Excluded** (full reasoning in `w03_feature_leakage_check.ipynb` Section 4): `fact_content_query_90d` (different grain/window), `provider_used`/`model_used` (operational metadata), `content_updated_date`/`last_optimized_date`/`optimization_eligible_date` (unverifiable point-in-time risk), `is_deleted` (filter only, never a feature), all hash-ID columns (grouping only). Public-safe: no client names, URLs, or raw query text appear anywhere in this pipeline — every reference is a pseudonymous hash ID from the warehouse itself.

In [ ]:
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.ga4_sessions) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS sessions_per_impression,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate,
        SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS ai_referral_share,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days,
        MAX(d.word_count) AS word_count,
        MAX(d.search_volume) AS search_volume,
        MAX(d.competition) AS competition,
        MAX(d.content_type) AS content_type,
        MAX(d.main_intent) AS main_intent
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4' AND d.is_deleted IS FALSE AND d.is_published IS TRUE
    GROUP BY 1, 2
""").df()
raw = raw.dropna(subset=['avg_ctr', 'avg_position'])
print(f"Rows: {len(raw)}")
print(f"Distinct clients: {raw['client_hash_id'].nunique()}")
print(f"Date window: {MID_MONTH} only (mid-panel); {FINAL_MONTH} sealed and unused for features/labels")

## 3. Methodology

**Target (proxy)**: `high_performer = avg_ctr > median(avg_ctr)`, thresholded within the mid-panel month. Unsupervised-adjacent proxy — no ground-truth archetype/performance label exists in the warehouse.

**Features (12)**: `avg_position`, `total_impressions`, `sessions_per_impression`, `engagement_rate`, `ai_referral_share`, `content_age_days`, `word_count`, `search_volume`, `competition`, one-hot `content_type`/`main_intent`, and a KMeans (k=6) `archetype_cluster` label. `avg_ctr` itself excluded (defines the target — the leakage lesson from `w03`).

**Baseline**: `score = expected_ctr_for_position_bucket − actual_ctr`, a transparent single-signal rule.

**Validation design**: `GroupShuffleSplit` on `client_hash_id` (30% test), confirmed zero client overlap — avoids the model/rule learning client-specific editorial style instead of a generalizable pattern.

**Leakage checks performed**: (a) sealed-month deliberate-leak trap (`w03`, repeated on final feature set in `w06`); (b) naive-random-split vs. grouped-split comparison (`w06`); (c) confirmation no excluded/operational fields reached the final feature set.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

ratio_cols = ['sessions_per_impression', 'engagement_rate', 'ai_referral_share']
raw[ratio_cols] = raw[ratio_cols].fillna(0)
raw['word_count'] = raw['word_count'].fillna(raw['word_count'].median())
raw['search_volume'] = raw['search_volume'].fillna(0)
raw['competition'] = raw['competition'].fillna(raw['competition'].median())
raw['content_type'] = raw['content_type'].fillna('unknown')
raw['main_intent'] = raw['main_intent'].fillna('unknown')
raw['high_performer'] = (raw['avg_ctr'] > raw['avg_ctr'].median()).astype(int)

cluster_input_cols = ['avg_position', 'total_impressions', 'sessions_per_impression', 'engagement_rate',
                       'ai_referral_share', 'content_age_days', 'word_count', 'search_volume', 'competition']
scaler = StandardScaler()
X_cluster = scaler.fit_transform(raw[cluster_input_cols])
raw['archetype_cluster'] = KMeans(n_clusters=6, random_state=0, n_init=10).fit_predict(X_cluster).astype(str)
print(f"high_performer base rate: {raw['high_performer'].mean():.1%}  (median split -> ~50% by design)")

## 4. Results (vs. baseline)

**Headline numbers, already run in `w05_model.ipynb` / `w06_validation_audit.ipynb`:**

| Method | AUC |
|---|---|
| Week-4 baseline rule (single signal) | **0.963** *(near-circular — `corr(baseline_score, avg_ctr) = -1.000`, see Limitations)* |
| Week-5 Random Forest (full features + archetype cluster) | **0.927** *(honest — independent of the target by construction)* |

**Combined `archetype_cluster` permutation importance: 0.0013** — effectively zero. The cluster feature did not meaningfully help prediction.

**Deliberate-leak trap on final feature set** (`w06`): AUC with `future_clicks` leaked in = 0.925, AUC honest = 0.927 — the leak did *not* inflate the score this time (see Limitations for why this differs from the `w03` trap).

**Error profile**: confusion matrix `[[19784, 1308], [3159, 8195]]`, 13.8% misclassified on a 32,446-row test set. Dominant permutation-importance features: `total_impressions` (0.153), `sessions_per_impression` (0.074).

In [ ]:
# Rebuild the two headline numbers here so the paper's claims are checkable from this notebook directly
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

position_bins = [0, 3, 10, 20, 50, float('inf')]
position_labels = ['1-3', '4-10', '11-20', '21-50', '51+']
raw['position_bucket'] = pd.cut(raw['avg_position'], bins=position_bins, labels=position_labels)

cat_cols = ['content_type', 'main_intent', 'archetype_cluster']
encoded = pd.get_dummies(raw, columns=cat_cols, prefix=['ctype', 'intent', 'cluster'])
feature_cols = [c for c in encoded.columns
                 if c not in ('content_hash_id', 'client_hash_id', 'avg_ctr', 'high_performer', 'position_bucket')]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(encoded, groups=encoded['client_hash_id']))
train_df, test_df = encoded.iloc[train_idx], encoded.iloc[test_idx]

benchmark = train_df.groupby(raw.loc[train_df.index, 'position_bucket'], observed=True)['avg_ctr'].mean()
test_bucket = raw.loc[test_df.index, 'position_bucket']
baseline_score = test_bucket.map(benchmark) - test_df['avg_ctr']
baseline_auc = roc_auc_score(test_df['high_performer'], -baseline_score.fillna(0))

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0, class_weight='balanced')
rf.fit(train_df[feature_cols], train_df['high_performer'])
model_auc = roc_auc_score(test_df['high_performer'], rf.predict_proba(test_df[feature_cols])[:, 1])

print(f"Baseline AUC (this re-run): {baseline_auc:.3f}")
print(f"Model AUC (this re-run):    {model_auc:.3f}")
print("(Small differences from the headline table above are expected/fine — different random split draw;")
print("the direction and near-circularity finding should still hold.)")

## 5. Limitations

1. **Baseline AUC is inflated by near-circularity**, not genuine predictive skill — `corr(baseline_score, avg_ctr) = -1.000` on the held-out split, since the baseline score is a near-exact linear rescaling of the target itself. The model's 0.927, built from features independent of the target, is the more trustworthy number.
2. **Archetype clustering did not earn a predictive role** — 0.0013 combined importance is a genuine negative result, not a shortcoming to hide. Clusters likely describe content *type*, not performance.
3. **Single cross-sectional month** — no multi-month trend data used, so nothing here distinguishes "consistently underperforming" from "an unusually quiet month."
4. **No sealed holdout evaluation was actually run** — every reference to `month=2026-06` in this pipeline exists only to *demonstrate* leakage, never to produce a genuine blind evaluation. A claim of "evaluated once, blind" would not be checkable from this repo as-is.
5. **Observational only** — every relationship (position↔CTR, age↔CTR) is correlational; no causal design.
6. **Scope-limited** — `gsc_and_ga4`-access clients only, one month, one client mix; not a guarantee for a different month or the full client base.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Ranked recommendations

From `w07_action_playbook.ipynb`: item-level queue flags `CTR_BELOW_POSITION_BENCHMARK` → `REVIEW_TITLE_META`, scored only above a 30-impression noise floor. Archetype-level mapping assigns each of the 6 clusters one of `REFRESH_CANDIDATE`, `REVIEW_TITLE_META`, `DISTRIBUTE_SUPPORT`, or `MONITOR_ONLY` based on that cluster's own position/CTR/age profile — not hand-picked. Every recommendation is decision-support for a human editor, never an auto-apply action (full no-go list in `w07` Section 3).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 7. Artifacts the paper embeds

Files the deployed paper pulls from, all produced by earlier notebooks in this repo:

- `work/outputs/baseline_action_score.csv` (regenerated on run, not committed — CI leak-guard)
- `work/outputs/archetype_action_mapping.csv` (committed reference table, from `w07`)
- `work/outputs/w07_metrics.json` (committed receipts: benchmark by position bucket, drift check, decay check, from `w07`)
- `work/figures/ctr_by_position_bucket.png` (committed figure, from `w07`)
- This notebook's own Section 4 output (baseline AUC, model AUC, printed above) — the numbers the paper's Results section states

In [ ]:
import os
print("Artifact check (run after w07 has been executed at least once in this environment):")
for path in ['work/outputs/archetype_action_mapping.csv', 'work/outputs/w07_metrics.json',
             'work/figures/ctr_by_position_bucket.png']:
    print(f"  {path}: {'FOUND' if os.path.exists(path) else 'MISSING — re-run w07 in this same session first'}")